# Notebook 03 — Metadata and Schema Audit

## Purpose
I inspect every column in every table: dtype, cardinality, sample values, and
key/foreign-key relationships.  This is the detailed schema audit.

## Why this matters
Understanding the relational structure prevents incorrect joins.  For example,
`customer_id` in orders is NOT the same as `customer_unique_id` — missing this
would break the RFM analysis.

## Inputs
`data/raw/` CSV files

## Outputs
`reports/tables/schema_audit.csv`

## Decisions made here
- Confirm join keys are consistent across tables
- Identify columns that will need encoding
- Flag any foreign-key violations (orders with no matching items, etc.)


In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.config import load_config
from src.paths import Paths
from src.preprocessing import check_schema, EXPECTED_SCHEMAS
from src.utils import cardinality_summary, save_table

cfg   = load_config()
paths = Paths(cfg)

# Reload the key tables
orders      = pd.read_csv(paths.olist_file('orders', cfg),
                          parse_dates=cfg['preprocessing']['date_columns'])
items       = pd.read_csv(paths.olist_file('order_items', cfg))
customers   = pd.read_csv(paths.olist_file('customers', cfg))
products    = pd.read_csv(paths.olist_file('products', cfg))
reviews     = pd.read_csv(paths.olist_file('order_reviews', cfg))
payments    = pd.read_csv(paths.olist_file('order_payments', cfg))
sellers     = pd.read_csv(paths.olist_file('sellers', cfg))
translation = pd.read_csv(paths.olist_file('category_translation', cfg))


In [2]:
# Run schema checks for all tables
print("Running schema checks...")
for name, df in [('orders', orders), ('order_items', items),
                 ('order_payments', payments), ('order_reviews', reviews),
                 ('customers', customers), ('products', products),
                 ('sellers', sellers)]:
    missing = check_schema(df, name)


Running schema checks...
  [schema] 'orders' OK — all 8 expected columns present
  [schema] 'order_items' OK — all 7 expected columns present
  [schema] 'order_payments' OK — all 5 expected columns present
  [schema] 'order_reviews' OK — all 3 expected columns present
  [schema] 'customers' OK — all 5 expected columns present
  [schema] 'products' OK — all 3 expected columns present
  [schema] 'sellers' OK — all 4 expected columns present


In [3]:
# Cardinality audit for orders table
print("=== Orders table cardinality ===")
display(cardinality_summary(orders))


=== Orders table cardinality ===


,column,dtype,n_unique,pct_unique,sample
0,order_id,object,99441,100.00,"['e481f51cbdc54678b7cc49136f2d6af7', '53cdb2fc..."
1,customer_id,object,99441,100.00,"['9ef432eb6251297304e76186b10a928d', 'b0830fb4..."
2,order_status,object,8,0.01,"['delivered', 'invoiced', 'shipped']"
3,order_purchase_timestamp,datetime64[ns],98875,99.43,"[Timestamp('2017-10-02 10:56:33'), Timestamp('..."
4,order_approved_at,datetime64[ns],90733,91.24,"[Timestamp('2017-10-02 11:07:15'), Timestamp('..."
5,order_delivered_carrier_date,datetime64[ns],81018,81.47,"[Timestamp('2017-10-04 19:55:00'), Timestamp('..."
6,order_delivered_customer_date,datetime64[ns],95664,96.20,"[Timestamp('2017-10-10 21:25:13'), Timestamp('..."
7,order_estimated_delivery_date,datetime64[ns],459,0.46,"[Timestamp('2017-10-18 00:00:00'), Timestamp('..."


In [4]:
# Key relationship verification
print("=== Key relationship checks ===")
print()

# orders -> items
order_ids_in_orders = set(orders['order_id'])
order_ids_in_items  = set(items['order_id'])
orphan_items = order_ids_in_items - order_ids_in_orders
print(f"Orders in items but NOT in orders table: {len(orphan_items)}")

# orders -> customers
cust_ids_in_orders = set(orders['customer_id'])
cust_ids_in_cust   = set(customers['customer_id'])
orphan_orders = cust_ids_in_orders - cust_ids_in_cust
print(f"customer_ids in orders but NOT in customers table: {len(orphan_orders)}")

# orders -> reviews
order_ids_in_reviews = set(reviews['order_id'])
orders_without_review = order_ids_in_orders - order_ids_in_reviews
print(f"Orders with NO review: {len(orders_without_review)} ({len(orders_without_review)/len(orders)*100:.1f}%)")

# products -> translation
cat_names = set(products['product_category_name'].dropna())
trans_keys = set(translation['product_category_name'])
untranslated = cat_names - trans_keys
print(f"Product categories with no English translation: {len(untranslated)}")
if untranslated:
    print(f"  Examples: {list(untranslated)[:5]}")


=== Key relationship checks ===

Orders in items but NOT in orders table: 0
customer_ids in orders but NOT in customers table: 0
Orders with NO review: 768 (0.8%)
Product categories with no English translation: 2
  Examples: ['portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer']


In [5]:
# Unique customer analysis: customer_id vs customer_unique_id
print("=== Customer ID analysis ===")
print(f"customer_id (per order):      {customers['customer_id'].nunique():,}")
print(f"customer_unique_id (per person): {customers['customer_unique_id'].nunique():,}")
print()
# Count how many unique customers placed multiple orders
orders_with_unique = orders.merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left'
)
orders_per_unique = orders_with_unique.groupby('customer_unique_id')['order_id'].count()
multi_order_customers = (orders_per_unique > 1).sum()
print(f"Customers with >1 order: {multi_order_customers:,} ({multi_order_customers/len(orders_per_unique)*100:.1f}%)")


=== Customer ID analysis ===
customer_id (per order):      99,441
customer_unique_id (per person): 96,096

Customers with >1 order: 2,997 (3.1%)


In [6]:
# Review score distribution
print("=== Review scores ===")
rc = reviews['review_score'].value_counts().sort_index()
for score, count in rc.items():
    bar = '#' * int(count / rc.max() * 30)
    print(f"  {score} star: {count:>6,}  {bar}")
print(f"  Mean: {reviews['review_score'].mean():.3f}")
print(f"  Std:  {reviews['review_score'].std():.3f}")


=== Review scores ===
  1 star: 11,424  #####
  2 star:  3,151  #
  3 star:  8,179  ####
  4 star: 19,142  ##########
  5 star: 57,328  ##############################
  Mean: 4.086
  Std:  1.348


In [7]:
# Product categories audit
print("=== Product categories ===")
print(f"Unique categories (Portuguese): {products['product_category_name'].nunique()}")
print(f"Translation table rows:         {len(translation)}")
print()
merged_cats = products.merge(translation, on='product_category_name', how='left')
missing_trans = merged_cats['product_category_name_english'].isna().sum()
print(f"Products with missing English category: {missing_trans} ({missing_trans/len(products)*100:.1f}%)")
print()
print("Top 10 categories by product count:")
print(products['product_category_name'].value_counts().head(10))


=== Product categories ===
Unique categories (Portuguese): 73
Translation table rows:         71

Products with missing English category: 623 (1.9%)

Top 10 categories by product count:
product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64


In [8]:
# Save schema audit
schema_rows = []
for name, df in [('orders', orders), ('items', items), ('payments', payments),
                 ('reviews', reviews), ('customers', customers),
                 ('products', products), ('sellers', sellers)]:
    for col in df.columns:
        schema_rows.append({
            'table': name, 'column': col,
            'dtype': str(df[col].dtype),
            'n_unique': df[col].nunique(),
            'n_null': df[col].isnull().sum(),
            'pct_null': round(df[col].isnull().mean() * 100, 2),
            'sample_values': str(df[col].dropna().unique()[:3].tolist()),
        })
schema_df = pd.DataFrame(schema_rows)
save_table(schema_df, 'schema_audit',
           reports_dir=str(paths.reports_tabs),
           paper_dir=str(paths.paper_tabs))
print()
print("Notebook 03 complete. Schema audit saved.")


  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\reports\tables\schema_audit.csv
  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\paper_or_report\tables\schema_audit.csv

Notebook 03 complete. Schema audit saved.
